# Sentiment API — Exploratory Data Analysis

This notebook documents the data exploration and model analysis process behind the Sentiment-Aware Text Classification API.

**Sections:**
1. Dataset Exploration — class balance, text length distribution, sample reviews
2. Tokenization Walkthrough — what DistilBERT actually sees
3. Training Curves — loss over training steps (simulated; see note in cell)
4. Evaluation — confusion matrix and live predictions

In [ ]:
import sys
sys.path.append('..')  # make project root importable from notebooks/

import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

---
## 1. Dataset Exploration

In [ ]:
dataset = load_dataset('imdb')

train_data = dataset['train']
test_data  = dataset['test']

print(f'Train samples : {len(train_data):,}')
print(f'Test samples  : {len(test_data):,}')
print(f'Features      : {train_data.features}')

In [ ]:
# Class distribution — confirming the dataset is balanced
train_labels = train_data['label']
neg_count = train_labels.count(0)
pos_count = train_labels.count(1)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['NEGATIVE', 'POSITIVE'], [neg_count, pos_count], color=['#e07b7b', '#7babe0'], width=0.5)
ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=11)
ax.set_title('Class Distribution — IMDB Training Set', fontsize=13)
ax.set_ylabel('Sample Count')
ax.set_ylim(0, neg_count * 1.15)
plt.tight_layout()
plt.show()

print(f'\nClass balance: {neg_count / len(train_labels):.1%} negative / {pos_count / len(train_labels):.1%} positive')
print('Dataset is perfectly balanced — accuracy is a valid metric here.')

In [ ]:
# Text length distribution — justifies our MAX_LENGTH = 256 choice
# Approximating token count with word count (tokens ≈ words * 1.3 for English)
word_counts = [len(text.split()) for text in train_data['text']]

p50  = np.percentile(word_counts, 50)
p90  = np.percentile(word_counts, 90)
p95  = np.percentile(word_counts, 95)
p99  = np.percentile(word_counts, 99)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(word_counts, bins=80, color='#7babe0', edgecolor='white', linewidth=0.4)
for pct, val, color in [(50, p50, '#2ecc71'), (90, p90, '#f39c12'), (95, p95, '#e74c3c'), (99, p99, '#8e44ad')]:
    ax.axvline(val, color=color, linestyle='--', linewidth=1.5, label=f'p{pct} = {val:.0f} words')
ax.set_title('Word Count Distribution — IMDB Training Set', fontsize=13)
ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Reviews')
ax.legend()
plt.tight_layout()
plt.show()

print(f'p50  : {p50:.0f} words')
print(f'p90  : {p90:.0f} words')
print(f'p95  : {p95:.0f} words')
print(f'p99  : {p99:.0f} words')
print(f'\nMAX_LENGTH=256 tokens covers the vast majority of reviews.')
print('Tokens ≈ words × 1.3 for English, so 256 tokens ≈ ~197 words of capacity.')

In [ ]:
# Sample reviews — getting a feel for the data quality
print('=== Sample NEGATIVE reviews ===')
for ex in train_data.filter(lambda x: x['label'] == 0).select(range(2)):
    print(f'\n[NEGATIVE] {ex["text"][:300]}...')

print('\n=== Sample POSITIVE reviews ===')
for ex in train_data.filter(lambda x: x['label'] == 1).select(range(2)):
    print(f'\n[POSITIVE] {ex["text"][:300]}...')

In [ ]:
# HTML artifacts in the raw data — motivates the clean_text() step in the pipeline
raw_sample = train_data[0]['text']
html_tags = re.findall(r'<[^>]+>', raw_sample)

print('Raw text excerpt (first 400 chars):')
print(raw_sample[:400])
print(f'\nHTML tags found: {set(html_tags)}')
print('\nThese would consume token slots without carrying sentiment signal.')
print('clean_text() strips them before tokenization.')

---
## 2. Tokenization Walkthrough

DistilBERT does not read words — it reads integer token IDs. This section shows what the model actually receives as input.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

sample_text = 'This film was absolutely wonderful!'
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f'Raw text : {sample_text}')
print(f'Tokens   : {tokens}')
print(f'Token IDs: {token_ids}')
print(f'\nVocabulary size: {tokenizer.vocab_size:,}')
print('Note: DistilBERT lowercases all text (-uncased variant).')

In [ ]:
# Full encoding — what the model.forward() call actually receives
encoded = tokenizer(
    sample_text,
    padding='max_length',
    truncation=True,
    max_length=16,  # short for readability
    return_tensors='pt'
)

print('input_ids    :', encoded['input_ids'].tolist())
print('attention_mask:', encoded['attention_mask'].tolist())
print()
print('Special tokens:')
print(f'  [CLS] = {tokenizer.cls_token_id}  (start of sequence — classification reads this position)')
print(f'  [SEP] = {tokenizer.sep_token_id}  (end of sequence marker)')
print(f'  [PAD] = {tokenizer.pad_token_id}  (padding — attention_mask=0 tells model to ignore these)')

In [ ]:
# Subword tokenization — how DistilBERT handles unknown or rare words
unusual_text = 'The cinematography was breathtakingly uncharacteristic.'
unusual_tokens = tokenizer.tokenize(unusual_text)

print(f'Text   : {unusual_text}')
print(f'Tokens : {unusual_tokens}')
print()
print('## prefix marks a continuation of the previous token (subword unit).')
print('This lets DistilBERT handle words outside its vocabulary by breaking them into known pieces.')

---
## 3. Training Curves

> **Note:** The curves below are simulated to illustrate expected training behavior.
> To plot real curves, run `make train` first. DistilBERT on IMDB typically converges
> to ~0.17 loss and ~93% accuracy within 3 epochs.

In [ ]:
# Simulated training loss — realistic curve shape for DistilBERT fine-tuning on IMDB
# 25,000 training samples / batch size 16 = 1,562 steps per epoch
steps_per_epoch = 1562
total_steps = steps_per_epoch * 3
x = np.arange(1, total_steps + 1)

np.random.seed(42)
base_loss = 0.65 * np.exp(-x / (total_steps * 0.45)) + 0.17
noise = np.random.normal(0, 0.015, size=len(x))
simulated_loss = base_loss + noise

# Smooth with a rolling average for the trend line
window = 60
smoothed = np.convolve(simulated_loss, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x, simulated_loss, color='#aec6e8', linewidth=0.6, alpha=0.7, label='Step loss')
ax.plot(x[window-1:], smoothed, color='#2471a3', linewidth=2.0, label='Smoothed (60-step avg)')

for epoch in range(1, 4):
    ax.axvline(epoch * steps_per_epoch, color='#e74c3c', linestyle='--', linewidth=1.0,
               label=f'Epoch {epoch} end' if epoch == 1 else None)

ax.set_title('Training Loss — DistilBERT Fine-tuning on IMDB (Simulated)', fontsize=13)
ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('../model/artifacts/training_curves_simulated.png', dpi=150)
plt.show()
print('Loss starts near 0.65 (close to random for binary cross-entropy) and decays to ~0.17 over 3 epochs.')

---
## 4. Evaluation

> **Run `make train` and `make evaluate` before executing these cells.**

In [ ]:
from pathlib import Path
from IPython.display import Image, display

cm_path = Path('../model/artifacts/confusion_matrix.png')

if cm_path.exists():
    display(Image(filename=str(cm_path)))
else:
    print('Confusion matrix not found. Run `make evaluate` to generate it.')

In [ ]:
from pathlib import Path

artifact_path = Path('../model/artifacts/final')

if not artifact_path.exists():
    print('No trained model found at model/artifacts/final.')
    print('Run `make train` from the project root first.')
else:
    from api.predictor import Predictor

    predictor = Predictor()

    test_inputs = [
        'One of the best films I have ever seen. A true masterpiece.',
        'Terrible movie. Boring, poorly acted, and a complete waste of time.',
        'It had its moments but ultimately fell flat.',
        'The acting was phenomenal and the story kept me on the edge of my seat.',
        'I walked out after 20 minutes. Absolutely dreadful.',
    ]

    print(f'{"Text":<60} {"Label":<12} {"Confidence"}')
    print('-' * 85)
    for text in test_inputs:
        result = predictor.predict(text)
        print(f'{text[:58]:<60} {result["label"]:<12} {result["confidence"]:.4f}')